# 🎓 استودیو بینایی سه‌بعدی | 3D Vision Studio
### سرور بازسازی مدل سه‌بعدی ابری با پردازشگر گرافیکی Google Colab T4
### مدل: **OpenAI Shap-E** (`openai/shap-e-img2img`)

---
### 🌟 ویژگی‌های مدل OpenAI Shap-E:
- **پایداری ۱۰۰٪ و بدون نیاز به کامپایل**: نصب سریع در چند ثانیه بدون خطای CUDA و بدون پیش‌نیازهای پیچیده.
- **بدون نیاز به لاگین یا توکن Hugging Face**: به صورت مستقیم و عمومی در دسترس است.
- **تولید سریع مش ۳۶۰ درجه**: تولید فایل خروجی استاندارد `.glb` در کمتر از ۵ الی ۸ ثانیه روی پردازشگر T4.

### 📌 راهنمای اجرا:
1. ابتدا مطمئن شوید به پردازشگر گرافیکی متصل هستید:
   - از منوی بالا: **Runtime** -> **Change runtime type** -> انتخاب **T4 GPU** -> کلیک روی **Save**.
2. از منو روی **Runtime** -> **Run all** کلیک کنید.
3. در سلول آخر (Cell 4)، لینک عمومی امن Cloudflare (مانند `https://xxxx.trycloudflare.com`) نمایش داده می‌شود.
4. آن آدرس را کپی کرده و در بخش تنظیمات سرور وب‌اپلیکیشن (Colab Settings) وارد کنید!

In [ ]:
# Step 1: Verify NVIDIA GPU & Install Standard Packages
!nvidia-smi

print("[*] Installing standard Hugging Face packages (Diffusers, Accelerate, FastAPI, Cloudflared)...")
!pip install -q diffusers transformers accelerate trimesh fastapi uvicorn python-multipart

# Download cloudflared tunnel binary for instant public HTTPS endpoint
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("[+] All packages and tools installed successfully in seconds!")

In [ ]:
# Step 2: Initialize OpenAI Shap-E Model on GPU
import torch
from diffusers import ShapEImg2ImgPipeline
from PIL import Image

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"[*] Loading OpenAI Shap-E model onto: {device}...")

pipe = ShapEImg2ImgPipeline.from_pretrained(
    "openai/shap-e-img2img",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    variant="fp16" if torch.cuda.is_available() else None,
).to(device)

gpu_title = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"[+] OpenAI Shap-E Model loaded successfully on: {gpu_title}!")

In [ ]:
# Step 3: Define FastAPI Endpoints (OpenAI Shap-E Pipeline)
import io
import time
import trimesh
from PIL import Image
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import Response
from diffusers.utils import export_to_obj

app = FastAPI(title="3D Vision Studio API (OpenAI Shap-E)")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/health")
def health_check():
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    vram_alloc = torch.cuda.memory_allocated(0) / (1024 ** 3) if torch.cuda.is_available() else 0.0
    vram_total = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3) if torch.cuda.is_available() else 0.0
    return {
        "status": "ok",
        "model": "OpenAI Shap-E (openai/shap-e-img2img)",
        "gpu_name": gpu_name,
        "vram_allocated_gb": round(vram_alloc, 2),
        "vram_total_gb": round(vram_total, 2),
        "model_ready": True,
        "timestamp": time.time()
    }

@app.post("/api/generate")
async def generate_3d(image: UploadFile = File(...)):
    try:
        contents = await image.read()
        pil_img = Image.open(io.BytesIO(contents)).convert("RGB").resize((256, 256))

        # Neural 3D synthesis with OpenAI Shap-E
        images = pipe(
            pil_img,
            guidance_scale=3.0,
            num_inference_steps=64,
            output_type="mesh"
        ).images
        mesh = images[0]

        # Export mesh to temp obj and convert to standard GLB
        temp_obj = "/tmp/shap_e_output.obj"
        export_to_obj(mesh, temp_obj)

        scene = trimesh.load(temp_obj)
        glb_io = io.BytesIO()
        scene.export(glb_io, file_type="glb")

        return Response(
            content=glb_io.getvalue(),
            media_type="model/gltf-binary",
            headers={"Content-Disposition": 'attachment; filename="model.glb"'}
        )
    except Exception as e:
        print(f"[!] Generation error: {e}")
        raise HTTPException(status_code=500, detail=str(e))

print("[+] FastAPI Application defined.")

In [ ]:
# Step 4: Launch FastAPI Server & Expose via Cloudflare Tunnel
import subprocess
import threading
import time
import re
import uvicorn

# Start Uvicorn in background thread
def run_api():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_api, daemon=True)
server_thread.start()
time.sleep(2)
print("[+] Uvicorn server listening on port 8000.")

# Launch Cloudflared tunnel
print("[*] Launching secure Cloudflare public tunnel...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

tunnel_url = None
for line in iter(tunnel_proc.stdout.readline, ""):
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break

if tunnel_url:
    print("\n" + "="*60)
    print("🎉 SUCCESS! YOUR COLAB BACKEND IS ONLINE!")
    print(f"👉 COPY THIS URL INTO YOUR WEB APP:\n{tunnel_url}")
    print("="*60 + "\n")
else:
    print("[!] Cloudflare tunnel did not output URL yet. Check output above.")